# 05A — Finalize Static Features

**FYP:** Developing a Semi-Dynamic Landslide Susceptibility Decision-Support Dashboard Using Machine Learning for Malaysia

This notebook closes Stage 05A.

It uses the outputs from:

- `05A_static_feature_extraction.ipynb`
- `05A_QC_soil_lithology.ipynb`

and applies the final corrections before moving to temporal features.

## Soil decision

ERA5-Land / GLDAS provide dynamic soil moisture and temperature, not historical soil texture. Therefore:

- SoilGrids remains the source for clay, sand, silt, and bulk density.
- SoilGrids is treated as a quasi-static conditioning factor.
- ERA5-Land soil moisture can be added later in Stage 05B as a time-varying hydrological feature.
- Missing/all-zero SoilGrids values are not replaced with Malaysia-wide averages.

## Geology decision

The JMG geology service contains overlapping polygons. This notebook aggregates structured geology attributes across all polygons covering each sample rather than taking the first arbitrary match.


In [1]:
from pathlib import Path
import math
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio

from pyproj import Transformer

warnings.filterwarnings("ignore", category=UserWarning)

print("pandas:", pd.__version__)
print("geopandas:", gpd.__version__)
print("rasterio:", rasterio.__version__)


pandas: 3.0.2
geopandas: 1.1.4
rasterio: 1.5.0


## 1. Project paths

In [2]:
PROJECT_ROOT = Path(r"C:\Landslide_FYP")

RAW_DIR = PROJECT_ROOT / "Landslide_FYP_Data"
DEV_DIR = PROJECT_ROOT / "Landslide_FYP_Development"

QC_DIR = DEV_DIR / "03_derived_features" / "static_features_qc"
STATIC_DIR = DEV_DIR / "03_derived_features" / "static_features"

SAMPLES_QC_PATH = QC_DIR / "static_feature_samples_qc.geojson"
LITHOLOGY_PATH = STATIC_DIR / "jmg_lithology_all_layers.geojson"

SOIL_ROOT = RAW_DIR / "03_Factors" / "Soil"

SOIL_PATHS = {
    "soil_clay": SOIL_ROOT / "Clay" / "soilgrids_clay_malaysia.tif",
    "soil_sand": SOIL_ROOT / "Sand" / "soilgrids_sand_malaysia.tif",
    "soil_silt": SOIL_ROOT / "Silt" / "soilgrids_silt_malaysia.tif",
    "soil_bulk_density": SOIL_ROOT / "Bulk_Density" / "soilgrids_bdod_malaysia.tif",
}

OUTPUT_DIR = DEV_DIR / "03_derived_features" / "static_features_final"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

required = {
    "QC sample table": SAMPLES_QC_PATH,
    "JMG lithology polygons": LITHOLOGY_PATH,
    **SOIL_PATHS,
}

for name, path in required.items():
    if not path.exists():
        raise FileNotFoundError(f"{name} not found:\n{path}")
    print("FOUND:", name, "->", path)

print("\nOutput folder:")
print(OUTPUT_DIR)


FOUND: QC sample table -> C:\Landslide_FYP\Landslide_FYP_Development\03_derived_features\static_features_qc\static_feature_samples_qc.geojson
FOUND: JMG lithology polygons -> C:\Landslide_FYP\Landslide_FYP_Development\03_derived_features\static_features\jmg_lithology_all_layers.geojson
FOUND: soil_clay -> C:\Landslide_FYP\Landslide_FYP_Data\03_Factors\Soil\Clay\soilgrids_clay_malaysia.tif
FOUND: soil_sand -> C:\Landslide_FYP\Landslide_FYP_Data\03_Factors\Soil\Sand\soilgrids_sand_malaysia.tif
FOUND: soil_silt -> C:\Landslide_FYP\Landslide_FYP_Data\03_Factors\Soil\Silt\soilgrids_silt_malaysia.tif
FOUND: soil_bulk_density -> C:\Landslide_FYP\Landslide_FYP_Data\03_Factors\Soil\Bulk_Density\soilgrids_bdod_malaysia.tif

Output folder:
C:\Landslide_FYP\Landslide_FYP_Development\03_derived_features\static_features_final


## 2. Load QC samples and JMG geology polygons

In [3]:
samples = gpd.read_file(SAMPLES_QC_PATH).to_crs("EPSG:4326")
lithology = gpd.read_file(LITHOLOGY_PATH).to_crs("EPSG:4326")

print("Samples:", len(samples))
print("JMG polygons:", len(lithology))

display(
    samples["sample_class"]
    .value_counts()
    .rename_axis("sample_class")
    .reset_index(name="records")
)


Samples: 3200
JMG polygons: 18742


,sample_class,records
0,background_candidate,3000
1,landslide,200


# PART A — FINAL SOIL HANDLING

## 3. Identify soil-invalid samples

All-four-zero SoilGrids samples are treated as invalid/background artefacts unless recovered from a nearby valid pixel.


In [4]:
soil_cols = [
    "soil_clay",
    "soil_sand",
    "soil_silt",
    "soil_bulk_density",
]

for col in soil_cols:
    samples[col] = pd.to_numeric(samples[col], errors="coerce")

samples["soil_all_zero"] = (
    samples[soil_cols]
    .fillna(0)
    .eq(0)
    .all(axis=1)
)

samples["soil_missing_any"] = samples[soil_cols].isna().any(axis=1)

samples["soil_needs_recovery"] = (
    samples["soil_all_zero"]
    | samples["soil_missing_any"]
)

display(
    samples.groupby(
        ["sample_class", "soil_needs_recovery"]
    )
    .size()
    .reset_index(name="records")
)

print(
    "Total samples needing soil recovery:",
    int(samples["soil_needs_recovery"].sum())
)


,sample_class,soil_needs_recovery,records
0,background_candidate,False,2930
1,background_candidate,True,70
2,landslide,False,122
3,landslide,True,78


Total samples needing soil recovery: 148


## 4. Nearest-valid SoilGrids recovery

Recovery is attempted only for **positive landslide samples**.

The search is limited to nearby pixels and never uses a national average.


In [5]:
MAX_SOIL_SEARCH_PIXELS = 5

def nearest_valid_raster_value(
    raster_path,
    lon,
    lat,
    max_pixels=5,
    invalid_zero=True
):
    with rasterio.open(raster_path) as src:

        transformer = Transformer.from_crs(
            "EPSG:4326",
            src.crs,
            always_xy=True
        )

        x, y = transformer.transform(lon, lat)
        row, col = src.index(x, y)

        best = None

        for radius in range(max_pixels + 1):

            r0 = max(0, row - radius)
            r1 = min(src.height, row + radius + 1)
            c0 = max(0, col - radius)
            c1 = min(src.width, col + radius + 1)

            window = rasterio.windows.Window(
                c0,
                r0,
                c1 - c0,
                r1 - r0
            )

            data = src.read(1, window=window)

            for rr in range(data.shape[0]):
                for cc in range(data.shape[1]):

                    value = data[rr, cc]

                    if not np.isfinite(value):
                        continue

                    if src.nodata is not None and value == src.nodata:
                        continue

                    if invalid_zero and value == 0:
                        continue

                    global_row = r0 + rr
                    global_col = c0 + cc

                    pixel_distance = math.sqrt(
                        (global_row - row) ** 2
                        + (global_col - col) ** 2
                    )

                    if best is None or pixel_distance < best["pixel_distance"]:
                        best = {
                            "value": float(value),
                            "pixel_distance": float(pixel_distance)
                        }

            if best is not None:
                break

        if best is None:
            return {
                "value": np.nan,
                "pixel_distance": np.nan
            }

        return best


## 5. Recover positive samples

In [6]:
samples["soil_recovery_attempted"] = False
samples["soil_recovery_success"] = False
samples["soil_recovery_max_pixel_distance"] = np.nan

positive_problem_mask = (
    samples["sample_class"].eq("landslide")
    & samples["soil_needs_recovery"]
)

print(
    "Positive landslides requiring soil recovery:",
    int(positive_problem_mask.sum())
)

recovery_rows = []

for idx in samples.index[positive_problem_mask]:

    lon = float(samples.at[idx, "longitude"])
    lat = float(samples.at[idx, "latitude"])

    samples.at[idx, "soil_recovery_attempted"] = True

    success_all = True
    recovered_distances = []

    for feature_name, raster_path in SOIL_PATHS.items():

        current_value = samples.at[idx, feature_name]

        if pd.notna(current_value) and current_value != 0:
            continue

        result = nearest_valid_raster_value(
            raster_path,
            lon,
            lat,
            max_pixels=MAX_SOIL_SEARCH_PIXELS,
            invalid_zero=True
        )

        if pd.notna(result["value"]):
            samples.at[idx, feature_name] = result["value"]
            recovered_distances.append(result["pixel_distance"])
        else:
            success_all = False

        recovery_rows.append({
            "sample_id": samples.at[idx, "sample_id"],
            "feature": feature_name,
            "original_value": current_value,
            "recovered_value": result["value"],
            "pixel_distance": result["pixel_distance"],
            "success": pd.notna(result["value"]),
        })

    samples.at[idx, "soil_recovery_success"] = success_all

    if recovered_distances:
        samples.at[
            idx,
            "soil_recovery_max_pixel_distance"
        ] = max(recovered_distances)

soil_recovery_audit = pd.DataFrame(recovery_rows)

display(soil_recovery_audit.head(30))

soil_recovery_audit.to_csv(
    OUTPUT_DIR / "positive_soil_recovery_audit.csv",
    index=False
)


Positive landslides requiring soil recovery: 78


,sample_id,feature,original_value,recovered_value,pixel_distance,success
0,POS_0003,soil_clay,0,394.0,1.000000,True
1,POS_0003,soil_sand,0,309.0,1.000000,True
2,POS_0003,soil_silt,0,296.0,1.000000,True
3,POS_0003,soil_bulk_density,0,95.0,1.000000,True
4,POS_0007,soil_clay,0,404.0,4.472136,True
5,POS_0007,soil_sand,0,314.0,4.472136,True
6,POS_0007,soil_silt,0,282.0,4.472136,True
7,POS_0007,soil_bulk_density,0,131.0,4.123106,True
8,POS_0008,soil_clay,0,408.0,2.828427,True
9,POS_0008,soil_sand,0,290.0,2.828427,True


## 6. Recalculate final soil validity

Background candidates are not repaired. Invalid ones can be excluded later because the candidate pool is large.


In [7]:
samples["soil_final_all_zero"] = (
    samples[soil_cols]
    .fillna(0)
    .eq(0)
    .all(axis=1)
)

samples["soil_final_missing_any"] = samples[soil_cols].isna().any(axis=1)

samples["soil_valid_final"] = (
    ~samples["soil_final_all_zero"]
    & ~samples["soil_final_missing_any"]
)

soil_final_summary = (
    samples.groupby(
        ["sample_class", "soil_valid_final"]
    )
    .size()
    .reset_index(name="records")
)

display(soil_final_summary)

soil_final_summary.to_csv(
    OUTPUT_DIR / "soil_final_validity_summary.csv",
    index=False
)


,sample_class,soil_valid_final,records
0,background_candidate,False,70
1,background_candidate,True,2930
2,landslide,False,6
3,landslide,True,194


# PART B — STRUCTURED JMG GEOLOGY

## 7. Prepare geology fields

The following JMG fields are retained when available:

- `AGE`
- `AGECODE`
- `GLN`
- `GLL`
- `FNM`


In [8]:
geo_fields = [
    "AGE",
    "AGECODE",
    "GLN",
    "GLL",
    "FNM",
]

available_geo_fields = [
    col for col in geo_fields
    if col in lithology.columns
]

print(
    "Available geology fields:",
    available_geo_fields
)


Available geology fields: ['AGE', 'AGECODE', 'GLN', 'GLL', 'FNM']


## 8. Spatially join samples to all overlapping geology polygons


In [9]:
join_cols = (
    available_geo_fields
    + [
        "_jmg_layer_id",
        "_jmg_layer_name",
        "geometry",
    ]
)

geo_join = gpd.sjoin(
    samples[
        ["sample_id", "geometry"]
    ],
    lithology[join_cols],
    how="left",
    predicate="within"
)

print("Spatial join rows:", len(geo_join))
print("Unique samples:", geo_join["sample_id"].nunique())


Spatial join rows: 6368
Unique samples: 3200


## 9. Aggregate structured geology attributes per sample


In [10]:
def clean_unique_values(series):
    values = []

    for value in series:
        if pd.isna(value):
            continue

        text = str(value).strip()

        if not text or text.lower() in {
            "nan",
            "none",
            "null",
            "<na>",
        }:
            continue

        if text not in values:
            values.append(text)

    return values


def aggregate_field(group, field):
    values = clean_unique_values(group[field])

    if not values:
        return pd.NA

    return " | ".join(values)


agg_rows = []

for sample_id, group in geo_join.groupby("sample_id"):

    row = {
        "sample_id": sample_id,
        "geology_polygon_count": int(
            group["_jmg_layer_id"].notna().sum()
        ),
    }

    if "AGE" in available_geo_fields:
        row["geology_age"] = aggregate_field(group, "AGE")

    if "AGECODE" in available_geo_fields:
        row["geology_age_code"] = aggregate_field(group, "AGECODE")

    if "GLN" in available_geo_fields:
        row["geology_unit_name"] = aggregate_field(group, "GLN")

    if "GLL" in available_geo_fields:
        row["geology_unit_code"] = aggregate_field(group, "GLL")

    if "FNM" in available_geo_fields:
        row["geology_unit_type"] = aggregate_field(group, "FNM")

    layer_names = clean_unique_values(
        group["_jmg_layer_name"]
    )

    row["geology_source_layers"] = (
        " | ".join(layer_names)
        if layer_names
        else pd.NA
    )

    agg_rows.append(row)

geology_features = pd.DataFrame(agg_rows)

display(geology_features.head(20))


,sample_id,geology_polygon_count,geology_age,geology_age_code,geology_unit_name,geology_unit_code,geology_unit_type,geology_source_layers
0,BG_00001,2,Ordovician-Silurian,8000.0,NaN,NaN,Geological Lithostratigraphic Unit,Lithology of Kedah | Rock age of Kedah
1,BG_00002,2,Igneous Activities: Triassic,9500.0,NaN,NaN,Geological Lithodemic Unit,Lithology of Kedah | Rock age of Kedah
2,BG_00003,2,Ordovician-Silurian,8000.0,NaN,NaN,Geological Lithostratigraphic Unit,Lithology of Kedah | Rock age of Kedah
3,BG_00004,2,Quaternary,1000.0,NaN,NaN,Geological Lithostratigraphic Unit,Lithology of Kedah | Rock age of Kedah
4,BG_00005,2,Ordovician-Silurian,8000.0,NaN,NaN,Geological Lithostratigraphic Unit,Lithology of Kedah | Rock age of Kedah
5,BG_00006,2,Quaternary,1000.0,NaN,NaN,Geological Lithostratigraphic Unit,Lithology of Perak | Rock age of Perak
6,BG_00007,2,Quaternary,1000.0,NaN,NaN,Geological Lithostratigraphic Unit,Lithology of Perak | Rock age of Perak
7,BG_00008,2,Quaternary,1000.0,NaN,NaN,Geological Lithostratigraphic Unit,Lithology of Perak | Rock age of Perak
8,BG_00009,2,Quaternary,1000.0,NaN,NaN,Geological Lithostratigraphic Unit,Lithology of Pulau_Pinang | Rock age of Pulau_...
9,BG_00010,2,Quaternary,1000.0,NaN,NaN,Geological Lithostratigraphic Unit,Lithology of Pulau_Pinang | Rock age of Pulau_...


## 10. Merge structured geology into samples

In [11]:
drop_old = [
    "lithology_raw",
    "_jmg_layer_id",
    "_jmg_layer_name",
    "lithology_polygon_matches",
    "lithology_available",
]

samples = samples.drop(
    columns=[
        c for c in drop_old
        if c in samples.columns
    ],
    errors="ignore"
)

samples = samples.merge(
    geology_features,
    on="sample_id",
    how="left"
)

samples = gpd.GeoDataFrame(
    samples,
    geometry="geometry",
    crs="EPSG:4326"
)

samples["geology_spatially_covered"] = (
    samples["geology_polygon_count"]
    .fillna(0)
    .gt(0)
)

structured_cols = [
    c for c in [
        "geology_age",
        "geology_age_code",
        "geology_unit_name",
        "geology_unit_code",
        "geology_unit_type",
    ]
    if c in samples.columns
]

if structured_cols:
    samples["geology_attribute_available"] = (
        samples[structured_cols]
        .notna()
        .any(axis=1)
    )
else:
    samples["geology_attribute_available"] = False

print(
    "Spatially covered:",
    int(samples["geology_spatially_covered"].sum()),
    "/",
    len(samples)
)

print(
    "Structured geology attribute available:",
    int(samples["geology_attribute_available"].sum()),
    "/",
    len(samples)
)


Spatially covered: 3166 / 3200
Structured geology attribute available: 3166 / 3200


# PART C — FINAL STATIC MODEL READINESS

## 11. Final national baseline validity

Required:
- valid terrain
- valid SoilGrids after QC/recovery
- major-fault distance

Geology remains optional/enhanced unless final attribute coverage is complete enough.


In [12]:
samples["national_core_static_valid_final"] = (
    samples["terrain_valid"]
    & samples["soil_valid_final"]
    & samples["fault_distance_available"]
)

samples["geology_enhanced_valid_final"] = (
    samples["national_core_static_valid_final"]
    & samples["geology_spatially_covered"]
    & samples["geology_attribute_available"]
)

final_validity = pd.DataFrame({
    "check": [
        "Total samples",
        "Landslide positives",
        "Background candidates",
        "Terrain valid",
        "Soil valid final",
        "Fault distance available",
        "National core-static valid final",
        "Geology spatially covered",
        "Geology attribute available",
        "Geology-enhanced valid final",
    ],
    "value": [
        len(samples),
        int(samples["sample_class"].eq("landslide").sum()),
        int(samples["sample_class"].eq("background_candidate").sum()),
        int(samples["terrain_valid"].sum()),
        int(samples["soil_valid_final"].sum()),
        int(samples["fault_distance_available"].sum()),
        int(samples["national_core_static_valid_final"].sum()),
        int(samples["geology_spatially_covered"].sum()),
        int(samples["geology_attribute_available"].sum()),
        int(samples["geology_enhanced_valid_final"].sum()),
    ]
})

display(final_validity)

final_validity.to_csv(
    OUTPUT_DIR / "final_static_validity_summary.csv",
    index=False
)


,check,value
0,Total samples,3200
1,Landslide positives,200
2,Background candidates,3000
3,Terrain valid,3200
4,Soil valid final,3124
5,Fault distance available,3200
6,National core-static valid final,3124
7,Geology spatially covered,3166
8,Geology attribute available,3166
9,Geology-enhanced valid final,3098


## 12. Export remaining invalid positive samples


In [13]:
invalid_positive = samples.loc[
    samples["sample_class"].eq("landslide")
    & ~samples["national_core_static_valid_final"]
].copy()

print(
    "Positive landslides still invalid:",
    len(invalid_positive)
)

display(
    invalid_positive[
        [
            "sample_id",
            "terrain_valid",
            "soil_valid_final",
            "fault_distance_available",
            "soil_recovery_attempted",
            "soil_recovery_success",
        ]
    ]
)

invalid_positive.drop(
    columns="geometry",
    errors="ignore"
).to_csv(
    OUTPUT_DIR / "positive_static_invalid_after_recovery.csv",
    index=False
)


Positive landslides still invalid: 6


,sample_id,terrain_valid,soil_valid_final,fault_distance_available,soil_recovery_attempted,soil_recovery_success
49,POS_0069,True,False,True,True,False
66,POS_0093,True,False,True,True,False
79,POS_0118,True,False,True,True,False
104,POS_0163,True,False,True,True,False
144,POS_0229,True,False,True,True,False
158,POS_0246,True,False,True,True,False


## 13. Save valid background candidates


In [14]:
valid_background = samples.loc[
    samples["sample_class"].eq("background_candidate")
    & samples["national_core_static_valid_final"]
].copy()

print(
    "Final valid background candidates:",
    len(valid_background)
)

valid_background.to_file(
    OUTPUT_DIR / "background_candidates_static_valid_final.geojson",
    driver="GeoJSON"
)

valid_background.drop(
    columns="geometry",
    errors="ignore"
).to_csv(
    OUTPUT_DIR / "background_candidates_static_valid_final.csv",
    index=False
)


Final valid background candidates: 2930


## 14. Save final static feature table


In [15]:
final_geojson = OUTPUT_DIR / "static_feature_samples_final.geojson"
final_csv = OUTPUT_DIR / "static_feature_samples_final.csv"

samples.to_file(
    final_geojson,
    driver="GeoJSON"
)

samples.drop(
    columns="geometry",
    errors="ignore"
).to_csv(
    final_csv,
    index=False
)

print("Saved:", final_geojson)
print("Saved:", final_csv)


Saved: C:\Landslide_FYP\Landslide_FYP_Development\03_derived_features\static_features_final\static_feature_samples_final.geojson
Saved: C:\Landslide_FYP\Landslide_FYP_Development\03_derived_features\static_features_final\static_feature_samples_final.csv


# PART D — FUTURE HISTORICAL SOIL MOISTURE

## 15. ERA5-Land / GLDAS note

Do not use these to replace SoilGrids texture.

Use them later as time-varying predictors:

- soil moisture
- soil temperature

Recommended Earth Engine datasets:

```python
ECMWF/ERA5_LAND/MONTHLY_AGGR
ECMWF/ERA5_LAND/DAILY_AGGR
NASA/GLDAS/V021/NOAH/G025/T3H
```

For this project, ERA5-Land is preferred for later soil-moisture work because it has finer spatial resolution than GLDAS and a long consistent time series.

This belongs in Stage 05B alongside rainfall.


In [16]:
ERA5_LAND_MONTHLY = "ECMWF/ERA5_LAND/MONTHLY_AGGR"
ERA5_LAND_DAILY = "ECMWF/ERA5_LAND/DAILY_AGGR"
GLDAS_21 = "NASA/GLDAS/V021/NOAH/G025/T3H"

print("Preferred later dynamic soil source:")
print(ERA5_LAND_MONTHLY)

print("\nAlternative:")
print(GLDAS_21)


Preferred later dynamic soil source:
ECMWF/ERA5_LAND/MONTHLY_AGGR

Alternative:
NASA/GLDAS/V021/NOAH/G025/T3H


# Stage 05A completion outputs

Please send:

```text
final_static_validity_summary.csv
positive_static_invalid_after_recovery.csv
soil_final_validity_summary.csv
static_feature_samples_final.csv
```

If the positive-invalid file is empty or very small and the final validity counts are sensible, Stage 05A is complete and the next notebook can be:

`05B_temporal_feature_extraction.ipynb`
